# Random Forest experiment: NVDA vs TSLA spread prediction

Build the modelling dataset using existing `src` pipeline (same logic as the logistic strategy):
- Load raw data → filter NVDA/TSLA → add daily returns
- Add accounting and technical indicators (not in raw data)
- Form pair spread and compute target `y` (sign of next-day spread)

No train/test split or scaling here; that will be handled later in the backtest loop.

In [2]:
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
import pandas as pd
from src.engine import make_dataset

In [4]:
# Build pair dataset: daily returns, accounting and technical features (all as NVDA - TSLA spread)
pair = make_dataset()
pair

,date,ret_a_next,ret_b_next,spread_next,spread_t,op_margin_diff,net_margin_diff,leverage_diff,cash_ratio_diff,rnd_intensity_diff,...,asset_turnover_diff,log_mcap_diff,mom5_diff,mom21_diff,mom63_diff,ma_gap20_diff,rsi14_diff,rv21_diff,vol_shock20_diff,y
0,2010-09-28,-0.007602,0.027103,-0.034705,-0.044378,1.127142,1.182227,-0.30658,0.155362,-0.283065,...,0.025449,0.25617,0.029987,0.096611,0.246499,0.087615,24.939359,-0.039620,-0.686257,0
1,2010-09-29,-0.016667,-0.071656,0.054989,-0.034705,1.127142,1.182227,-0.30658,0.155362,-0.283065,...,0.025449,0.221797,-0.06317,0.126175,0.241198,0.044127,22.026364,-0.090220,-0.751686,1
2,2010-09-30,-0.02842,0.009556,-0.037976,0.054989,1.127142,1.182227,-0.30658,0.155362,-0.283065,...,0.025449,0.279343,-0.037865,0.205277,0.196244,0.090149,25.694716,-0.171658,-0.963321,0
3,2010-10-01,-0.009692,0.018932,-0.028624,-0.037976,1.127142,1.182227,-0.30658,0.155362,-0.283065,...,0.025449,0.241001,-0.099101,0.199983,0.0344,0.040691,13.492571,-0.121212,0.537602,0
4,2010-10-04,0.007117,0.006193,0.000924,-0.028624,1.127142,1.182227,-0.30658,0.155362,-0.283065,...,0.025449,0.212507,-0.085349,0.177827,-0.194436,0.005487,13.318870,-0.111022,0.180264,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3834,2025-12-23,-0.003171,-0.00033,-0.002842,0.036537,0.565413,0.510896,-0.035944,0.06029,0.024518,...,0.14367,1.047068,0.073471,-0.183748,-0.027367,-0.016854,-3.985465,-0.072170,0.21527,0
3835,2025-12-24,0.01018,-0.021034,0.031214,-0.002842,0.565413,0.510896,-0.035944,0.06029,0.024518,...,0.14367,1.044221,0.064548,-0.128603,-0.084946,-0.015284,-6.064038,-0.020174,-0.371528,1
3836,2025-12-26,-0.012124,-0.032724,0.0206,0.031214,0.565413,0.510896,-0.035944,0.06029,0.024518,...,0.14367,1.075608,0.111043,-0.061486,-0.009684,0.020077,1.861026,-0.049022,0.042188,1
3837,2025-12-29,-0.003613,-0.011335,0.007722,0.0206,0.565413,0.510896,-0.035944,0.06029,0.024518,...,0.14367,1.096681,0.084752,-0.033284,-0.001985,0.04104,-4.227646,-0.068708,-0.209645,1


In [ ]:
# Date range and columns
print("Date range:", pair["date"].min(), "~", pair["date"].max())
pair.columns.tolist()

In [ ]:
# Preview the pair dataframe
pair.head()

In [ ]:
# Target y distribution (1 = NVDA beats TSLA next day, 0 = TSLA beats NVDA)
pair["y"].value_counts().sort_index()

In [5]:
# Optional: quick check for missing values in feature columns
from src import config
pair[config.FEATURES].isna().sum()

op_margin_diff          0
net_margin_diff         0
leverage_diff           0
cash_ratio_diff         0
rnd_intensity_diff      0
capex_intensity_diff    0
asset_turnover_diff     0
log_mcap_diff           0
mom5_diff               0
mom21_diff              0
mom63_diff              0
ma_gap20_diff           0
rsi14_diff              0
rv21_diff               0
vol_shock20_diff        0
dtype: int64